### Wrap the SQL side as a genuinely reusable Unity Catalog function

In [0]:
%sql
CREATE OR REPLACE FUNCTION policyiq.gold.get_compliance_status(
  p_branch_id STRING DEFAULT NULL,
  p_policy_domain STRING DEFAULT NULL,
  p_compliance_filter STRING DEFAULT 'ALL'
)
RETURNS TABLE (
  branch_id STRING, branch_name STRING, policy_domain STRING, kpi_name STRING,
  kpi_display_name STRING, policy_id STRING, policy_section_ref STRING,
  actual_value DOUBLE, threshold_value DOUBLE, threshold_operator STRING,
  unit STRING, compliance_status STRING, raw_gap DOUBLE
)
COMMENT 'Returns branch-level KPI compliance results with policy clause references. p_policy_domain accepts natural language like "KYC", "AML", "credit risk", "cybersecurity", "HR leave". p_compliance_filter accepts natural language like "non-compliant", "violations", "compliant", "ok", "all" - both are normalized internally, pass whatever wording the user used.'
RETURN
  WITH normalized AS (
    SELECT
      CASE
        WHEN p_policy_domain IS NULL THEN NULL
        WHEN lower(p_policy_domain) LIKE '%kyc%'                                    THEN 'ekyc'
        WHEN lower(p_policy_domain) LIKE '%aml%' OR lower(p_policy_domain) LIKE '%mltf%' OR lower(p_policy_domain) LIKE '%launder%' THEN 'aml_mltf'
        WHEN lower(p_policy_domain) LIKE '%credit%' OR lower(p_policy_domain) LIKE '%loan%'    THEN 'credit_risk'
        WHEN lower(p_policy_domain) LIKE '%cyber%' OR lower(p_policy_domain) LIKE '%security%' THEN 'cybersecurity'
        WHEN lower(p_policy_domain) LIKE '%hr%' OR lower(p_policy_domain) LIKE '%leave%'      THEN 'hr_leave'
        ELSE lower(p_policy_domain)
      END AS domain_norm,
      CASE
        WHEN lower(p_compliance_filter) LIKE '%non%'  OR lower(p_compliance_filter) LIKE '%violat%' OR lower(p_compliance_filter) LIKE '%fail%' THEN 'NON_COMPLIANT'
        WHEN lower(p_compliance_filter) LIKE '%comply%' OR lower(p_compliance_filter) LIKE '%compliant%' OR lower(p_compliance_filter) LIKE '%ok%' OR lower(p_compliance_filter) LIKE '%pass%' THEN 'COMPLIANT'
        ELSE 'ALL'
      END AS filter_norm
  )
  SELECT f.branch_id, f.branch_name, f.policy_domain, f.kpi_name, f.kpi_display_name,
         f.policy_id, f.policy_section_ref, f.actual_value, f.threshold_value,
         f.threshold_operator, f.unit, f.compliance_status, f.raw_gap
  FROM policyiq.silver.kpi_compliance_facts f, normalized n
  WHERE (p_branch_id IS NULL OR f.branch_id = p_branch_id)
    AND (n.domain_norm IS NULL OR f.policy_domain = n.domain_norm)
    AND (n.filter_norm = 'ALL'
         OR (n.filter_norm = 'NON_COMPLIANT' AND f.compliance_status = 'Non-Compliant')
         OR (n.filter_norm = 'COMPLIANT' AND f.compliance_status = 'Compliant'));

In [0]:
%sql
SELECT * FROM policyiq.gold.get_compliance_status(
  p_branch_id => NULL,
  p_policy_domain => 'KYC',
  p_compliance_filter => 'non-compliant'
);

In [0]:
%sql
CREATE OR REPLACE FUNCTION policyiq.gold.search_policy_text(
  p_query STRING,
  p_policy_domain STRING DEFAULT NULL
)
RETURNS TABLE (policy_id STRING, policy_name STRING, policy_section_ref STRING, chunk_text STRING)
COMMENT 'Semantic search over all policy documents. Use this to answer questions about what a policy requires, defines, or states, when you need the exact clause text rather than a compliance number.'
RETURN
  SELECT policy_id, policy_name, NULL AS policy_section_ref, chunk_text
  FROM VECTOR_SEARCH(
    index => 'policyiq.silver.policy_chunks_index',
    query => p_query,
    num_results => 5
  )
  WHERE p_policy_domain IS NULL OR policy_domain = p_policy_domain;

In [0]:
%sql
SELECT * FROM policyiq.gold.search_policy_text(
  p_query => 'related party lending limit as percentage of capital',
  p_policy_domain => 'credit_risk'
);

In [0]:
%sql
CREATE OR REPLACE FUNCTION policyiq.gold.list_all_violations(
  p_branch_id STRING DEFAULT NULL,
  p_policy_domain STRING DEFAULT NULL
)
RETURNS TABLE (
  branch_id STRING, branch_name STRING, policy_domain STRING, kpi_name STRING,
  kpi_display_name STRING, policy_id STRING, policy_section_ref STRING,
  actual_value DOUBLE, threshold_value DOUBLE, threshold_operator STRING,
  unit STRING, raw_gap DOUBLE
)
COMMENT 'Returns ONLY non-compliant KPI violations (never compliant rows) with policy clause citations. Use this any time the user asks to see violations, non-compliance, exceptions, or breaches - across all branches and all domains by default. Optionally filter to one branch_id or one policy_domain (accepts natural language like "KYC", "credit risk", "cybersecurity", "HR").'
RETURN
  WITH normalized AS (
    SELECT
      CASE
        WHEN p_policy_domain IS NULL THEN NULL
        WHEN lower(p_policy_domain) LIKE '%kyc%' THEN 'ekyc'
        WHEN lower(p_policy_domain) LIKE '%aml%' OR lower(p_policy_domain) LIKE '%mltf%' OR lower(p_policy_domain) LIKE '%launder%' THEN 'aml_mltf'
        WHEN lower(p_policy_domain) LIKE '%credit%' OR lower(p_policy_domain) LIKE '%loan%' THEN 'credit_risk'
        WHEN lower(p_policy_domain) LIKE '%cyber%' OR lower(p_policy_domain) LIKE '%security%' THEN 'cybersecurity'
        WHEN lower(p_policy_domain) LIKE '%hr%' OR lower(p_policy_domain) LIKE '%leave%' THEN 'hr_leave'
        ELSE lower(p_policy_domain)
      END AS domain_norm
  )
  SELECT f.branch_id, f.branch_name, f.policy_domain, f.kpi_name, f.kpi_display_name,
         f.policy_id, f.policy_section_ref, f.actual_value, f.threshold_value,
         f.threshold_operator, f.unit, f.raw_gap
  FROM policyiq.silver.kpi_compliance_facts f, normalized n
  WHERE f.compliance_status = 'Non-Compliant'
    AND (p_branch_id IS NULL OR f.branch_id = p_branch_id)
    AND (n.domain_norm IS NULL OR f.policy_domain = n.domain_norm);

In [0]:
%sql
SELECT count(*) AS total_violations FROM policyiq.gold.list_all_violations();